In [7]:
try:
    import kagglehub
except ImportError:
    !pip install kagglehub
try:
    import huggingface_hub
except ImportError:
    !pip install huggingface_hub
try:
    import ipywidgets
except ImportError:
    !pip install ipywidgets
    !pip install --upgrade ipywidgets

# !pip install --upgrade notebook jupyterlab

from src.dataset.DroneSegDataSet import MyDataset
from src.model.U_NetModel import U_Net
from src.model.MultiU_NetModel import MultiU_Net
from src.training.TrainSession import train_session
from src.dataset.CheckDataset import check_dataset

In [8]:
def print_model_params(model):
    print("Model parameter summary:")
    print("-" * 60)
    total = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            numel = param.numel()
            total += numel
            print(f"{name:<50} {numel:>12,}")
    print("-" * 60)
    print(f"{'Total trainable parameters':<50} {total:>12,}")


In [9]:
# 检查硬件环境
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = U_Net(
    in_channel=22,
    depth=2,
).to(device)

print("model parameters count")
print_model_params(model)

# print("Model summary: ")
# print(model)


Using device: cuda
model parameters count
Model parameter summary:
------------------------------------------------------------
inc.conv.0.depthwise.0.weight                               198
inc.conv.0.depthwise.0.bias                                  22
inc.conv.0.depthwise.1.weight                                22
inc.conv.0.depthwise.1.bias                                  22
inc.conv.0.pointwise.0.weight                             1,408
inc.conv.0.pointwise.0.bias                                  64
inc.conv.0.pointwise.1.weight                                64
inc.conv.0.pointwise.1.bias                                  64
inc.conv.1.depthwise.0.weight                               576
inc.conv.1.depthwise.0.bias                                  64
inc.conv.1.depthwise.1.weight                                64
inc.conv.1.depthwise.1.bias                                  64
inc.conv.1.pointwise.0.weight                             4,096
inc.conv.1.pointwise.0.bias             

In [ ]:
from torchvision import transforms
# 加载数据集
ds_path = check_dataset()
dataset = MyDataset(
    'drone_seg_dataset/classes_dataset/classes_dataset/original_images',
    'drone_seg_dataset/classes_dataset/classes_dataset/label_images_semantic',
    transform=transforms.Compose([
        transforms.ToTensor(),
    ])
)

from torch.utils.data import random_split, DataLoader
import numpy as np

train_set, test_set = random_split(dataset, [int(0.9 * len(dataset)), len(dataset) - int(0.9 * len(dataset))])
train_loader = DataLoader(dataset=train_set, batch_size=4, shuffle=True)
# test_loader = DataLoader(dataset=test_set, batch_size=2, shuffle=False)

print(f"数据集总样本数: {len(dataset)}")
print(f"训练集样本数: {len(train_set)}")
print(f"测试集样本数: {len(test_set)}")

img0, lbl0, _ = train_set.__getitem__(0)
print(f"Train set - img shape: {img0.shape}, lbl shape: {lbl0.shape}")
lbl_unique = torch.unique(lbl0)
print(f"Train set - unique labels in lbl0: {lbl_unique}")

# img0, lbl0, _ = test_set.__getitem__(0)
# print(f"Test set - img shape: {img0.shape}, lbl shape: {lbl0.shape}")
# lbl_unique = torch.unique(lbl0)
# print(f"Train set - unique labels in lbl0: {lbl_unique}")

img0, lbl0, _ = dataset.__getitem__(0)

print("图像1的形状:", img0.shape)
print("标签1的形状:", lbl0.shape)

img0_batched = img0.unsqueeze(0)
ans0 = model(img0_batched.to(device))
print("预训练模式模型输出的形状:", ans0.shape)

img0_batched = img0.unsqueeze(0)
ans0 = model(img0_batched.to(device))
print("训练模式模型输出的形状:", ans0.shape)

sample_output = torch.argmax(ans0, dim=1).detach().cpu().numpy()
print("模型输出的唯一值 (类别索引):", np.unique(sample_output))


In [ ]:
INIT_MODEL_PATH = "" # 在此处填入.pth文件路径以加载预训练权重，留空则从头开始训练
if INIT_MODEL_PATH:
    model.load_state_dict(torch.load(INIT_MODEL_PATH, map_location=device))
    print(f"已从 '{INIT_MODEL_PATH}' 加载模型参数。")
else:
    print("未指定预训练模型，从头开始训练。")

from src.dataset.DroneSegDataSet import convert_to_binary_label

# --- 定义损失函数和优化器 ---
criterion = torch.nn.BCEWithLogitsLoss()
# 优化器使用 SGD，学习率与动量动态调整

class_idx_list = [0, 1, 2, 3, 4]
depth_list = [2, 3, 4]

for class_idx in class_idx_list:
    for depth in depth_list:
        print(f"训练模型 - 类别索引: {class_idx}, 深度: {depth}")
        model = U_Net(in_channel=22, depth=depth).to(device)

        def convert(labels: torch.Tensor) -> torch.Tensor:
            bin_label = convert_to_binary_label(labels, class_idx) # 格式为 [B, H, W]，值为0或1
            # 需要增加 通道C 维度，变为 [B, 1, H, W]，以适配 BCEWithLogitsLoss 的输入要求
            bin_label = bin_label.unsqueeze(1).float()  # 转换为 float 类型
            return bin_label.to(device)

        train_session(
            model=model,
            dataloader=train_loader,
            criterion=criterion,
            title=f"U-Net Class {class_idx} Depth {depth}",
            device=device,
            label_transform=convert,  # 将标签转换为二分类格式
        )
